In [1]:
# bibliotecques
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Chemin d'accès 
ROOT = Path.cwd().parent 
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
print(f"Project root: {ROOT}")

from src.data_loader import *
from src.periodicity import *
from src.hull_white import *
from src.sinusoidal_hw import *
from src.monte_carlo import *
from src.calibration import *
print("Imports OK")

Project root: C:\Users\Manuel\Desktop\sinusoidal_hw
Imports OK


In [5]:
# ---------------------------------------------------------------
#  On force les données exactes du papier (Table 4) pour réaliser une vraie comparaison 
# ---------------------------------------------------------------

# 1. Données "Hardcodées" d'après le PDF
maturities = [1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 20.0, 30.0]

# Valeurs exactes de la colonne "Observed (%)" de la Table 4 
# 1Y=1.22, 2Y=1.75, 3Y=1.91, 5Y=1.96, 7Y=2.01, 10Y=2.00, 20Y=2.45, 30Y=2.36
paper_yields_percent = [1.22, 1.75, 1.91, 1.96, 2.01, 2.00, 2.45, 2.36]

# Conversion en décimales pour le modèle
yields = [y / 100.0 for y in paper_yields_percent]
r0 = yields[0] # 1.22% [cite: 217]

In [7]:
# 2. Initialisation du Calibrateur
calibrator = Calibrator(maturities, yields)

# 3. Calibration Standard Hull-White
# Le papier obtient kappa ~ 0.3164 [cite: 267]
print("\n--- 1. Calibration HW Standard ---")
res_hw = calibrator.calibrate_standard_hw(r0)
print(f"Kappa obtenu : {res_hw['kappa']:.4f} (Papier attendu: ~0.3164)")
print(f"RMSE obtenu  : {res_hw['rmse']:.6f} (Papier attendu: ~0.0014)")

# 4. Calibration Sinusoidal Hull-White
# Le papier fixe omega à 0.00078 [cite: 203]
OMEGA_PAPER = 0.00078 

print(f"\n--- 2. Calibration Sinusoidal HW (Omega={OMEGA_PAPER}) ---")
res_sin = calibrator.calibrate_sinusoidal_hw(r0, fixed_omega=OMEGA_PAPER)


--- 1. Calibration HW Standard ---
Kappa obtenu : 0.3164 (Papier attendu: ~0.3164)
RMSE obtenu  : 0.007490 (Papier attendu: ~0.0014)

--- 2. Calibration Sinusoidal HW (Omega=0.00078) ---
Début calibration Sinusoidal HW (Monte Carlo, omega=0.000780)...


In [11]:
# 5. Affichage des Résultats Comparatifs
import pandas as pd
df_results = pd.DataFrame([
    ("Standard HW", res_hw['kappa'], res_hw['rmse']),
    ("Sinusoidal HW", res_sin['kappa_0'], res_sin['rmse'])
], columns=["Modèle", "Kappa (k/k0)", "RMSE"])

print("\n--- RESULTATS FINAUX (Doivent être proches du papier) ---")
display(df_results)

# Vérification du Prix 30 ans (Le "Juge de Paix")
# Papier: Standard=0.47, Sinusoidal=0.43, Market=0.49 


--- RESULTATS FINAUX (Doivent être proches du papier) ---


,Modèle,Kappa (k/k0),RMSE
0,Standard HW,0.316377,0.00749
1,Sinusoidal HW,0.322090,0.00546


In [13]:
# Vérification du Prix 30 ans
# Papier: Standard=0.47, Sinusoidal=0.43, Market=0.49 

# Calcul Prix Standard
hw = HullWhiteModel(res_hw['kappa'], res_hw['theta'], res_hw['sigma'])
p_hw_30 = hw.price_zero_coupon(0, 30, r0)

# Calcul Prix Sinusoidal
sin_model = SinusoidalHullWhiteModel(res_sin['kappa_0'], res_sin['A'], OMEGA_PAPER, res_sin['theta'], res_sin['sigma'])
mc = MonteCarloPricer(sin_model, n_paths=500, dt=0.05)
p_sin_30 = mc.price_bond(r0, 30)

print(f"\n--- PRIX OBLIGATION 30 ANS (Target: ~0.49) ---")
print(f"Prix Marché (Table 4 data) : {np.exp(-0.0236 * 30):.4f}")
print(f"Prix Standard HW           : {p_hw_30:.4f}")
print(f"Prix Sinusoidal HW         : {p_sin_30:.4f}")


--- PRIX OBLIGATION 30 ANS (Target: ~0.49) ---
Prix Marché (Table 4 data) : 0.4926
Prix Standard HW           : 0.4857
Prix Sinusoidal HW         : 0.5034
